# Rossby number and the topographic PV gradient

## Aim

For shallow-water PV, `PV = (zeta + f) / h`. Neglecting horizontal gradients of relative vorticity gives

`grad(PV) = grad(f)/h - (zeta + f) grad(h)/h**2`.

With the **signed** Rossby number `Ro = zeta/f`, the topographic contribution is

`beta_topo = -f (1 + Ro) grad(h)/h**2`.

This notebook tests whether CE relative vorticity amplifies the same environmental slope while AE relative vorticity cancels it, and whether this helps explain the polarity-dependent tilt response. The existing `Ro` column is absolute-valued by `add_pv_gradient_terms`; this notebook deliberately recalculates `Ro_signed = w/f`. Here `w` is vertical relative vorticity, not vertical velocity.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'seacofs_tilt_tools.py').exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or one of its subfolders.')
CASE_ROOT = ANALYSIS_ROOT / 'case_studies'
for path in (ANALYSIS_ROOT, CASE_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import seacofs_tilt_tools as tilt
from case_study_tools import PVAlignmentConfig, add_pv_alignment_diagnostics

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 80)

## 1. Load data and construct signed Rossby-number diagnostics

In [ ]:
MAX_DEPTH_m = 3000.0
MIN_TILT_km = 5.0
ALIGNMENT_DEG = 30.0

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df, _ = tilt.load_tilt_tables(paths)
df = tilt.add_region_labels(df, grid)
df = tilt.add_pv_gradient_terms(df, grid, core_mean=True)

config = PVAlignmentConfig(
    dominance_factor=2.0, max_topo_depth_m=MAX_DEPTH_m,
    min_tilt_distance_km=MIN_TILT_km,
)
d = add_pv_alignment_diagnostics(df, config)

# add_pv_gradient_terms stores abs(w/f) as Ro; retain it but use the signed value here.
d['Ro_abs_existing'] = d.Ro
d['Ro_signed'] = d.w / d.f
d['absolute_vorticity_factor'] = 1.0 + d.Ro_signed
d['abs_vort_from_Ro'] = d.f * d.absolute_vorticity_factor
d['slope_mag'] = np.hypot(d.dhdx, d.dhdy)

# Counterfactual topographic PV gradient if relative vorticity were zero.
d['PV_grad_topo_fonly_x'] = -d.f * d.dhdx / d.h**2
d['PV_grad_topo_fonly_y'] = -d.f * d.dhdy / d.h**2
d['PV_grad_topo_fonly_mag'] = np.hypot(d.PV_grad_topo_fonly_x, d.PV_grad_topo_fonly_y)
d['vorticity_amplification'] = d.PV_grad_topo_mag / d.PV_grad_topo_fonly_mag
d['topo_plan_ratio_linear'] = d.PV_grad_topo_mag / d.PV_grad_plan_mag
d['topo_plan_ratio_fonly'] = d.PV_grad_topo_fonly_mag / d.PV_grad_plan_mag
d['aligned_30'] = d.dtheta_PV_grad <= ALIGNMENT_DEG

valid = d.replace([np.inf, -np.inf], np.nan).dropna(subset=[
    'Ro_signed', 'PV_grad_topo_mag', 'PV_grad_topo_fonly_mag',
    'PV_grad_plan_mag', 'dtheta_PV_grad', 'TiltDis',
])
valid = valid[(valid.h <= MAX_DEPTH_m) & valid.direction_valid].copy()

print(f'{len(valid):,} directional observations from {valid.Eddy.nunique():,} eddies')
display(valid.groupby('Cyc').agg(observations=('Eddy', 'size'), eddies=('Eddy', 'nunique')).astype(int))

## 2. Algebra and sign checks

The computed amplification should equal `abs(1 + Ro_signed)`. CEs should normally have positive signed Ro; AEs should normally have negative signed Ro. Values with `1 + Ro_signed <= 0` imply zero or reversed absolute vorticity and require individual inspection.

In [ ]:
check = np.abs(valid.vorticity_amplification - np.abs(valid.absolute_vorticity_factor))
print(f'Max amplification identity error: {check.max():.3e}')
print(f'Max abs-vorticity identity error: {(valid.abs_vort - valid.abs_vort_from_Ro).abs().max():.3e}')

sign_summary = valid.groupby('Cyc').agg(
    median_Ro=('Ro_signed', 'median'),
    q05_Ro=('Ro_signed', lambda x: x.quantile(.05)),
    q95_Ro=('Ro_signed', lambda x: x.quantile(.95)),
    median_factor=('absolute_vorticity_factor', 'median'),
    fraction_factor_nonpositive=('absolute_vorticity_factor', lambda x: (x <= 0).mean()),
)
display(sign_summary.round(3))
display(valid.loc[valid.absolute_vorticity_factor <= 0,
                  ['Eddy', 'Day', 'Cyc', 'lat', 'f', 'w', 'Ro_signed', 'absolute_vorticity_factor']]
        .sort_values('Ro_signed').head(30))

In [ ]:
eddy_ro = (valid.groupby(['Cyc', 'Eddy'], as_index=False)
           .agg(Ro_signed=('Ro_signed', 'median'),
                factor=('absolute_vorticity_factor', 'median'),
                amplification=('vorticity_amplification', 'median'),
                latitude=('lat', 'median'), observations=('Day', 'size')))

fig, axs = plt.subplots(1, 2, figsize=(9, 3.6), constrained_layout=True)
for cyc, color in [('AE', 'tab:red'), ('CE', 'tab:blue')]:
    part = eddy_ro[eddy_ro.Cyc == cyc]
    axs[0].hist(part.Ro_signed, bins=40, density=True, histtype='step', lw=2, color=color, label=cyc)
    axs[1].hist(part.amplification, bins=40, density=True, histtype='step', lw=2, color=color, label=cyc)
axs[0].axvline(0, color='0.4', lw=.8)
axs[0].axvline(-1, color='0.4', ls='--', lw=.8)
axs[0].set(xlabel='Eddy median signed Ro = zeta/f', ylabel='Density')
axs[1].axvline(1, color='0.4', ls='--', lw=.8)
axs[1].set(xlabel='Topographic amplification |1 + Ro|', ylabel='Density')
for ax in axs: ax.legend(frameon=False)

## 3. How much of the CE-AE PV-gradient contrast comes from relative vorticity?

Compare the observed topographic term with the counterfactual obtained from the same latitude, depth and slope but with `zeta = 0`. Medians are first calculated within each eddy so long tracks do not dominate.

In [ ]:
eddy_terms = (valid.groupby(['Cyc', 'Eddy'], as_index=False)
              .agg(full_topo=('PV_grad_topo_mag', 'median'),
                   fonly_topo=('PV_grad_topo_fonly_mag', 'median'),
                   plan=('PV_grad_plan_mag', 'median'),
                   slope=('slope_mag', 'median'), depth=('h', 'median'),
                   latitude=('lat', 'median'), Ro_signed=('Ro_signed', 'median')))
summary = eddy_terms.groupby('Cyc').agg(
    eddies=('Eddy', 'size'),
    median_full_topo=('full_topo', 'median'),
    median_fonly_topo=('fonly_topo', 'median'),
    median_plan=('plan', 'median'),
)
display(summary)
for term in ['full_topo', 'fonly_topo']:
    ratio = summary.loc['CE', f'median_{term}'] / summary.loc['AE', f'median_{term}']
    print(f'CE/AE ratio for {term}: {ratio:.2f}')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 3.6), constrained_layout=True)
for cyc, color in [('AE', 'tab:red'), ('CE', 'tab:blue')]:
    part = eddy_terms[eddy_terms.Cyc == cyc]
    axs[0].hist(np.log(part.fonly_topo), bins=35, density=True, histtype='step', lw=2, color=color, label=cyc)
    axs[1].hist(np.log(part.full_topo), bins=35, density=True, histtype='step', lw=2, color=color, label=cyc)
axs[0].set(title='Counterfactual: zeta = 0', xlabel='eddy median log topographic PV gradient', ylabel='Density')
axs[1].set(title='Full: zeta + f', xlabel='eddy median log topographic PV gradient', ylabel='Density')
for ax in axs: ax.legend(frameon=False)

## 4. Latitude dependence

Rossby number changes with latitude because `f` changes. However, the dimensional topographic coefficient is `f(1 + Ro) = f + zeta`; Ro should therefore be interpreted together with `f`, not alone.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
for cyc, color in [('AE', 'tab:red'), ('CE', 'tab:blue')]:
    part = eddy_terms[eddy_terms.Cyc == cyc]
    axs[0].scatter(part.latitude, part.Ro_signed, s=8, alpha=.25, color=color, label=cyc)
    axs[1].scatter(part.latitude, np.abs(part.Ro_signed + 1), s=8, alpha=.25, color=color, label=cyc)
    # Recover median dimensional absolute vorticity for the same eddies.
    av = valid[valid.Cyc == cyc].groupby('Eddy').abs_vort.median().reindex(part.Eddy)
    axs[2].scatter(part.latitude, np.abs(av), s=8, alpha=.25, color=color, label=cyc)
axs[0].set(xlabel='Latitude', ylabel='Signed Ro')
axs[1].set(xlabel='Latitude', ylabel='|1 + Ro|')
axs[2].set(xlabel='Latitude', ylabel='|f + zeta| (s^-1)')
for ax in axs: ax.legend(frameon=False)

## 5. Alignment across the full topographic-to-planetary transition

Do not preselect only topographic-dominated observations when looking for the transition. The key nondimensional predictor is `|beta_topo| / |beta_plan| = |f(1 + Ro)| |grad(h)| / (beta h)`. Random unsigned orientation gives a median mismatch of 90 degrees and `P(mismatch <= 30 degrees) = 16.7%`.

In [ ]:
transition = d.replace([np.inf, -np.inf], np.nan).dropna(subset=[
    'topo_plan_ratio', 'dtheta_PV_grad', 'Ro_signed', 'TiltDis',
]).copy()
transition = transition[(transition.h <= MAX_DEPTH_m) & transition.direction_valid]
transition['aligned_30'] = transition.dtheta_PV_grad <= ALIGNMENT_DEG

def binned_response(part, x='topo_plan_ratio', n_bins=15):
    part = part.copy()
    part['bin'] = pd.qcut(part[x], n_bins, duplicates='drop')
    return (part.groupby('bin', observed=True)
            .agg(x=(x, 'median'), angle=('dtheta_PV_grad', 'median'),
                 q25=('dtheta_PV_grad', lambda z: z.quantile(.25)),
                 q75=('dtheta_PV_grad', lambda z: z.quantile(.75)),
                 p_aligned=('aligned_30', 'mean'), observations=('Eddy', 'size'),
                 eddies=('Eddy', 'nunique')).reset_index(drop=True))

fig, axs = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for cyc, color in [('AE', 'tab:red'), ('CE', 'tab:blue')]:
    stats = binned_response(transition[transition.Cyc == cyc])
    axs[0].plot(stats.x, stats.angle, '-o', color=color, label=cyc)
    axs[0].fill_between(stats.x, stats.q25, stats.q75, color=color, alpha=.15)
    axs[1].plot(stats.x, 100 * stats.p_aligned, '-o', color=color, label=cyc)
axs[0].axhline(90, color='0.4', ls=':', lw=.8)
axs[1].axhline(100/6, color='0.4', ls=':', lw=.8)
for ax in axs:
    ax.axvline(0, color='0.5', lw=.8, label='equal contributions')
    ax.axvline(np.log(2), color='0.5', ls='--', lw=.8, label='topographic 2x')
    ax.set_xlabel('log(|topographic PV gradient| / |planetary PV gradient|)')
axs[0].set(ylabel='Median tilt-PV mismatch (degrees)', ylim=(0, 180))
axs[1].set(ylabel='Aligned observations (%)', ylim=(0, 100))
axs[0].legend(frameon=False)

## 6. Separate environmental slope from vorticity amplification

The full topographic term is the product of an environmental term and an eddy term: `B0 = |f| |grad(h)| / h**2` and `A = |1 + Ro|`. Plotting alignment against both prevents a stronger CE response from being attributed solely to either steeper sampled slopes or stronger relative vorticity.

In [ ]:
valid['log_B0'] = np.log(valid.PV_grad_topo_fonly_mag)
valid['log_A'] = np.log(valid.vorticity_amplification)

fig, axs = plt.subplots(2, 2, figsize=(9, 7), constrained_layout=True)
for i, (cyc, color) in enumerate([('AE', 'tab:red'), ('CE', 'tab:blue')]):
    part = valid[(valid.Cyc == cyc) & np.isfinite(valid.log_A)]
    for j, (x, label) in enumerate([('log_B0', 'log environmental slope term B0'),
                                     ('log_A', 'log vorticity amplification |1 + Ro|')]):
        stats = binned_response(part, x=x, n_bins=12)
        axs[i, j].plot(stats.x, 100 * stats.p_aligned, '-o', color=color)
        axs[i, j].axhline(100/6, color='0.4', ls=':', lw=.8)
        axs[i, j].set(title=cyc, xlabel=label, ylim=(0, 100))
        if j == 0: axs[i, j].set_ylabel('Aligned observations (%)')

In [ ]:
# Two-dimensional descriptive heatmaps. Blank cells contain too few unique eddies.
fig, axs = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for ax, cyc in zip(axs, ['AE', 'CE']):
    part = valid[(valid.Cyc == cyc) & np.isfinite(valid.log_A)].copy()
    part['B_bin'] = pd.qcut(part.log_B0, 5, labels=False, duplicates='drop')
    part['A_bin'] = pd.qcut(part.log_A, 5, labels=False, duplicates='drop')
    grouped = part.groupby(['A_bin', 'B_bin'], observed=True)
    probability = grouped.aligned_30.mean().unstack() * 100
    n_eddies = grouped.Eddy.nunique().unstack()
    probability = probability.where(n_eddies >= 10)
    image = ax.imshow(probability, origin='lower', aspect='auto', vmin=0, vmax=60, cmap='viridis')
    ax.set(title=cyc, xlabel='Environmental slope B0 quintile', ylabel='|1 + Ro| quintile')
fig.colorbar(image, ax=axs, label='Aligned observations (%)', fraction=.03, pad=.03)

## 7. Environmental matching

Bin eddies by latitude, depth and slope using common edges, then compare CE and AE topographic terms only inside environmental strata containing both polarities. This is a transparent first check before a formal matched or mixed-effects analysis.

In [ ]:
matched = eddy_terms.copy()
matched['lat_bin'] = pd.cut(matched.latitude, bins=np.linspace(matched.latitude.min(), matched.latitude.max(), 7), include_lowest=True)
matched['depth_bin'] = pd.cut(matched.depth, bins=np.linspace(matched.depth.min(), matched.depth.max(), 7), include_lowest=True)
matched['slope_bin'] = pd.qcut(matched.slope, 6, duplicates='drop')
strata = (matched.groupby(['lat_bin', 'depth_bin', 'slope_bin', 'Cyc'], observed=True)
          .agg(n=('Eddy', 'size'), full=('full_topo', 'median'), fonly=('fonly_topo', 'median'),
               Ro=('Ro_signed', 'median')).reset_index())
paired = strata.pivot(index=['lat_bin', 'depth_bin', 'slope_bin'], columns='Cyc')
paired.columns = [f'{metric}_{cyc}' for metric, cyc in paired.columns]
paired = paired.dropna(subset=['full_AE', 'full_CE'])
paired = paired[(paired.n_AE >= 5) & (paired.n_CE >= 5)]
paired['CE_AE_full_ratio'] = paired.full_CE / paired.full_AE
paired['CE_AE_fonly_ratio'] = paired.fonly_CE / paired.fonly_AE
display(paired[['n_AE', 'n_CE', 'CE_AE_full_ratio', 'CE_AE_fonly_ratio']])
print(f'Matched strata: {len(paired)}')
print(f'Median matched CE/AE full ratio: {paired.CE_AE_full_ratio.median():.2f}')
print(f'Median matched CE/AE f-only ratio: {paired.CE_AE_fonly_ratio.median():.2f}')

## 8. Interpretation safeguards and next model

1. `Ro_signed` and the full topographic PV gradient are mathematically coupled, so do not interpret their raw correlation as independent evidence. The counterfactual and factorized analyses are more informative.
2. Daily observations within an eddy are correlated. Add eddy-clustered bootstrap intervals or a mixed-effects model before making inferential claims.
3. Do not choose a PV or Ro threshold by maximising alignment in the same observations used for evaluation. Compare smooth and segmented models using held-out eddies.
4. The approximation `grad(zeta) approximately 0` creates an effective core-mean PV-gradient diagnostic. If gridded velocity fields are available, compare `grad(zeta)/h` with the retained planetary and topographic terms.
5. A formal model should predict continuous angular mismatch or aligned/not-aligned status from `log(B0)`, signed `Ro`, polarity and their interactions, with eddy and region grouping.